# Removation.AI — Colab Setup (v2.3)

**Run All** (`Ctrl+F9`). First run ~25 min | Repeat runs ~5 min.

### Fixes in v2.2 → v2.3
- **Room Changer broken (CLIPSeg silent load failure):** Changed from zip+`cp -r` to `git clone` with force-clean. `cp -r` into an existing dir creates a nested `ComfyUI-CLIPSeg/ComfyUI-CLIPSeg-main/` structure that ComfyUI cannot find — node loaded silently as missing.
- **Furniture Remover broken (MiniMax `__init__.py` missing):** `ComfyUI-MiniMax-Remover` Drive clone had previously failed (empty dir). Now force-cloned fresh each session via `NODES_FORCE_CLONE`; Drive-cached models are symlinked back in.
- **Style Transfer broken (RealESRGAN_x4.pth corrupt):** Added ZIP-structure integrity check for upscale `.pth` models before download step. Corrupt files (partial downloads / Drive FUSE errors) are deleted and re-downloaded automatically.


**Run All** (`Ctrl+F9`). First run ~25 min | Repeat runs ~5 min.

### Fixes in v2.1 → v2.2
- **Drive models invisible to ComfyUI** (`photon_v1`, `canny`, `RealESRGAN_x4` not found): ComfyUI copies real empty directories on install; the old symlink code silently skipped them with `if not exists`. Now force-replaces real dirs with symlinks and also copies models directly to ComfyUI paths as a fallback.

### Fixes in v2 → v2.1
- **Empty log / ComfyUI not starting**: added `python -u` (unbuffered output) so crashes appear immediately
- **Timeout too short**: increased from 270s to 600s (10 min)
- **sys.exit() crashes the kernel**: replaced with safe error display

### What v2 fixed (still applies)
| # | Root Cause | Fix |
|---|-----------|-----|
| 1 | `opencv-python` → `libGL` crash kills all controlnet_aux nodes | Force `opencv-python-headless` |
| 2 | Drive-cached `comfyui_controlnet_aux` skips fresh clone | Delete cached copy at start |
| 3 | String-replacement patch breaks on repo format changes | Append alias to file end |
| 4 | MLSD model in wrong folder (`controlnet/` not `controlnet/1.5/`) | Correct path + auto-migrate |


In [ ]:
# ╭─────────────────────────────────────────────────────╮
# │   REMOVATION.AI — GOOGLE COLAB SETUP  (v2.3)       │
# ╰─────────────────────────────────────────────────────╯

from google.colab import drive
import os, sys, re, time, shutil, subprocess, threading, requests

DRIVE_BASE   = '/content/drive/MyDrive/RemovationAI'
DRIVE_MODELS = f'{DRIVE_BASE}/models'
DRIVE_NODES  = f'{DRIVE_BASE}/custom_nodes'
DRIVE_COMFY  = f'{DRIVE_BASE}/ComfyUI'
COMFYUI_PATH = '/content/ComfyUI'
NODES_PATH   = f'{COMFYUI_PATH}/custom_nodes'

MODEL_SUBDIRS = [
    'checkpoints', 'controlnet', 'controlnet/1.5', 'loras', 'vae', 'clip', 'clip_vision',
    'upscale_models', 'unet', 'diffusion_models', 'style_models',
    'inpaint', 'sam2', 'depth_anything_v2', 'IC-Light',
    'grounding-dino', 'insightface', 'onnx', 'ipadapter', 'sams',
    'RMBG',
]

# ── STEP 1: Mount Drive ─────────────────────────────────────────
print('━'*55)
print('STEP 1/9  Mount Google Drive')
print('━'*55)
drive.mount('/content/drive')
for d in [DRIVE_BASE, DRIVE_MODELS, DRIVE_NODES]:
    os.makedirs(d, exist_ok=True)
for s in MODEL_SUBDIRS:
    target = f'{DRIVE_MODELS}/{s}'
    # os.path.exists() returns False for broken symlinks, so use lexists() instead.
    # lexists() returns True for any filesystem entry — file, dir, or broken symlink.
    # makedirs(exist_ok=True) only suppresses errors for real directories;
    # a file or broken symlink at the target path still causes FileExistsError.
    if os.path.lexists(target) and not os.path.isdir(target):
        print(f'⚠️  Removing stale entry at {target} (was: {"broken symlink" if os.path.islink(target) else "file"})')
        os.remove(target)
    try:
        os.makedirs(target, exist_ok=True)
    except FileExistsError:
        pass  # concurrent creation or resolved-symlink already present
print('✅ Drive mounted:', DRIVE_BASE)

# ── STEP 2: GPU check ─────────────────────────────────────────
print()
print('━'*55)
print('STEP 2/9  GPU Check')
print('━'*55)
if shutil.which('nvidia-smi') is None:
    print('❌ No GPU. Runtime -> Change runtime type -> T4 GPU -> Save')
    sys.exit(1)
r = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader,nounits'],
    capture_output=True, text=True
)
try:
    name, total, free = [x.strip() for x in r.stdout.strip().split(',')]
    print(f'✅ GPU: {name}  |  VRAM: {int(total)/1024:.1f} GB total  |  {int(free)/1024:.1f} GB free')
except (ValueError, ZeroDivisionError):
    print(f'✅ GPU detected (raw: {r.stdout.strip()!r})')

# ── STEP 3: Install ComfyUI ────────────────────────────────────
print()
print('━'*55)
print('STEP 3/9  Install ComfyUI')
print('━'*55)
if os.path.exists(f'{DRIVE_COMFY}/main.py'):
    print('Cached on Drive — copying to /content...')
    if os.path.exists(COMFYUI_PATH):
        os.system(f'rm -rf {COMFYUI_PATH}')
    os.system(f'cp -r {DRIVE_COMFY} {COMFYUI_PATH}')
    print('✅ ComfyUI ready from cache')
else:
    print('First time — cloning ComfyUI (~2 min)...')
    os.system(f'git clone https://github.com/comfyanonymous/ComfyUI.git {DRIVE_COMFY} -q')
    os.system(f'cp -r {DRIVE_COMFY} {COMFYUI_PATH}')
    print('✅ ComfyUI cloned and cached to Drive')

# FIX #2: Delete any Drive-cached comfyui_controlnet_aux to guarantee fresh clone.
# Also covers the case where Step 4 creates ckpts/ before the node loop runs —
# the dummy-dir detection in Step 5 handles that case separately.
_cached_aux = f'{COMFYUI_PATH}/custom_nodes/comfyui_controlnet_aux'
if os.path.exists(_cached_aux) and any(
    fn.endswith('.py') for _, _, files in os.walk(_cached_aux) for fn in files
) is False:
    shutil.rmtree(_cached_aux)
    print('🧹 Removed Drive-cached comfyui_controlnet_aux (no code, only models) — will force fresh clone')
elif os.path.exists(_cached_aux):
    shutil.rmtree(_cached_aux)
    print('🧹 Removed Drive-cached comfyui_controlnet_aux — will force fresh clone')

print('Installing Python requirements...')
os.system(f'pip install -q -r {COMFYUI_PATH}/requirements.txt')
os.system('pip install -q torchvision torchaudio transformers accelerate diffusers safetensors timm einops kornia')
os.system('pip install -q groundingdino-py 2>/dev/null || true')
os.system('pip install -q supervision ultralytics 2>/dev/null || true')
print('✅ Python requirements installed')

# FIX #1 Part A: opencv conflict → kills all comfyui_controlnet_aux nodes
print('🔧 Pre-fixing OpenCV conflict (root cause of M_LSD_Lines missing)...')
os.system('apt-get install -qq libgl1 libglib2.0-0 2>/dev/null || true')
os.system('pip uninstall -q -y opencv-python 2>/dev/null || true')
os.system('pip install -q opencv-python-headless 2>/dev/null || true')
print('✅ OpenCV conflict pre-resolved (headless only)')

# Symlink model dirs: Drive -> ComfyUI
# FIX v2.2: ComfyUI creates real empty dirs during install/copy. The old
# `if not os.path.exists(link)` check silently skipped those real dirs, so
# Drive models were never visible. Now we force-replace real dirs with symlinks.
comfy_models = f'{COMFYUI_PATH}/models'
os.makedirs(comfy_models, exist_ok=True)
for subdir in MODEL_SUBDIRS:
    src  = f'{DRIVE_MODELS}/{subdir}'
    link = f'{comfy_models}/{subdir}'
    os.makedirs(src, exist_ok=True)
    if '/' in subdir:
        # Parent dir already symlinked to Drive; sub-path is auto-visible — no extra symlink.
        continue
    if subdir == 'sams':
        if os.path.islink(link): os.unlink(link)
        os.makedirs(link, exist_ok=True)
    else:
        if os.path.islink(link):
            pass  # already a correct symlink
        else:
            if os.path.isdir(link):
                # Real dir — migrate any existing files to Drive, then replace with symlink
                for item in os.listdir(link):
                    s_item = os.path.join(link, item)
                    d_item = os.path.join(src, item)
                    if not os.path.exists(d_item):
                        shutil.move(s_item, d_item)
                shutil.rmtree(link)
            elif os.path.lexists(link):
                # Broken symlink or stale file at this path — remove before relinking
                os.remove(link)
            os.symlink(src, link)
print('✅ Model directories symlinked to Drive (real dirs replaced)')

# ── STEP 4: Pre-start fixes ───────────────────────────────────
print()
print('━'*55)
print('STEP 4/9  Pre-start fixes (critical path setup)')
print('━'*55)

sam2_src  = f'{DRIVE_MODELS}/sam2/sam2_hiera_small.pt'
gdino_src = f'{DRIVE_MODELS}/grounding-dino/groundingdino_swint_ogc.pth'
depth_src = f'{DRIVE_MODELS}/depth_anything_v2/depth_anything_v2_vits.pth'

# 4a. SAM2
for t in [
    f'{COMFYUI_PATH}/models/sams/sam2_hiera_small.pt',
    f'{NODES_PATH}/comfyui-impact-pack/models/sam2/sam2_hiera_small.pt',
    f'{NODES_PATH}/comfyui-segment-anything/models/sam2/sam2_hiera_small.pt',
]:
    os.makedirs(os.path.dirname(t), exist_ok=True)
    if not os.path.exists(t) and os.path.exists(sam2_src):
        shutil.copy2(sam2_src, t)
print('✅ SAM2 paths')

# 4b. GroundingDINO
for t in [
    f'{NODES_PATH}/comfyui-segment-anything/models/groundingdino/groundingdino_swint_ogc.pth',
    f'{NODES_PATH}/comfyui-impact-pack/models/grounding-dino/groundingdino_swint_ogc.pth',
]:
    os.makedirs(os.path.dirname(t), exist_ok=True)
    if not os.path.exists(t) and os.path.exists(gdino_src):
        shutil.copy2(gdino_src, t)
print('✅ GroundingDINO paths')

# 4c. Depth Anything — NOTE: this creates the ckpts/ dir before the node is cloned.
# The dummy-dir detection in Step 5 handles this correctly (backs up ckpts, clones, restores).
# DepthAnythingV2Preprocessor uses custom_hf_download which stores models at
# ckpts/{hf_repo}/{filename} = ckpts/depth-anything/Depth-Anything-V2-Small/depth_anything_v2_vits.pth
# NOT the flat ckpts/depth_anything_v2_vits.pth path. Populate both so the node
# finds the file without a HuggingFace network round-trip.
depth_ckpts_base = f'{NODES_PATH}/comfyui_controlnet_aux/ckpts'
depth_nested_dir = f'{depth_ckpts_base}/depth-anything/Depth-Anything-V2-Small'
depth_target_flat   = f'{depth_ckpts_base}/depth_anything_v2_vits.pth'
depth_target_nested = f'{depth_nested_dir}/depth_anything_v2_vits.pth'
os.makedirs(depth_nested_dir, exist_ok=True)
if os.path.exists(depth_src):
    if not os.path.exists(depth_target_flat):
        shutil.copy2(depth_src, depth_target_flat)
    if not os.path.exists(depth_target_nested):
        shutil.copy2(depth_src, depth_target_nested)
# Also expose via models/depth_anything_v2/ symlink (Drive-backed) in case
# the node is ever updated to use folder_paths instead of ckpts_dir.
depth_target = depth_target_flat  # keep for Step 7 retry reference
print('✅ Depth Anything path (flat + nested)')

# 4d. CLIPSeg — required by Impact Pack's CLIPSegDetectorProvider at runtime.
# MUST use git clone (not zip+cp): zip extraction puts files inside a nested dir
# (custom_nodes/ComfyUI-CLIPSeg/ComfyUI-CLIPSeg-main/__init__.py) which ComfyUI
# cannot find, so the node silently never loads.
# MUST force-clean: a stale or empty cached dir causes the same silent failure.
clipseg_path = f'{NODES_PATH}/ComfyUI-CLIPSeg'
if os.path.islink(clipseg_path):
    os.unlink(clipseg_path)
elif os.path.exists(clipseg_path):
    shutil.rmtree(clipseg_path)
print('⏬ CLIPSeg (git clone)...', end=' ', flush=True)
_ret = os.system(f'git clone -q https://github.com/biegert/ComfyUI-CLIPSeg.git {clipseg_path} 2>/dev/null')
if _ret == 0 and os.path.exists(f'{clipseg_path}/__init__.py'):
    req = f'{clipseg_path}/requirements.txt'
    if os.path.exists(req):
        os.system(f'pip install -q -r {req} 2>/dev/null || true')
    os.system('pip install -q transformers')
    print('✅')
else:
    # Fallback: zip download with explicit flattening (not cp -r)
    os.system(f'rm -rf {clipseg_path} /tmp/clipseg.zip /tmp/clipseg_ext/')
    print('(git unavailable — zip fallback)...', end=' ', flush=True)
    os.system('wget -q -O /tmp/clipseg.zip https://github.com/biegert/ComfyUI-CLIPSeg/archive/refs/heads/main.zip')
    os.system('unzip -q /tmp/clipseg.zip -d /tmp/clipseg_ext/')
    _extracted = '/tmp/clipseg_ext/ComfyUI-CLIPSeg-main'
    if os.path.exists(_extracted) and os.path.exists(f'{_extracted}/__init__.py'):
        shutil.copytree(_extracted, clipseg_path)
        os.system('pip install -q transformers')
        print('✅ (via zip)')
    else:
        print('❌ CLIPSeg install failed — Room Changer wall detection will not work')

# 4e. LaMa
lama_node_dir = f'{NODES_PATH}/comfyui-lama-remover/models'
lama_drive    = f'{DRIVE_NODES}/comfyui-lama-remover/models/big-lama.pt'
lama_target   = f'{lama_node_dir}/big-lama.pt'
os.makedirs(lama_node_dir, exist_ok=True)
if not os.path.exists(lama_target):
    if os.path.exists(lama_drive):
        shutil.copy2(lama_drive, lama_target)
        print('✅ LaMa from Drive cache')
    else:
        print('⏬ Downloading LaMa...', end=' ', flush=True)
        os.system(f'wget -q -c -O "{lama_drive}" "https://huggingface.co/smartywu/big-lama/resolve/main/big-lama.pt"')
        if os.path.exists(lama_drive):
            shutil.copy2(lama_drive, lama_target)
            print('✅')
        else:
            print('❌')
else:
    print('✅ LaMa ready')

# 4f. folder_paths.py patch
fp_file = f'{COMFYUI_PATH}/folder_paths.py'
if not os.path.exists(fp_file):
    print('⚠️  folder_paths.py missing — ComfyUI install may have failed')
else:
    with open(fp_file, 'r') as f:
        content = f.read()
    if '_removation_patch_' in content:
        content = content.split('# _removation_patch_')[0]
    lines = content.split('\n')
    new_lines = []
    patched = False
    for line in lines:
        new_lines.append(line)
        if (not patched and 'models_dir' in line and '=' in line
                and 'os.path' in line and not line.strip().startswith('#')):
            new_lines.append('# _removation_patch_ force real path for Impact Pack sams lookup')
            new_lines.append('models_dir = "/content/ComfyUI/models"')
            patched = True
    # Append IC-Light unet fix at END of module so folder_names_and_paths is
    # fully initialised. add_model_folder_path('unet', models/) makes ComfyUI
    # resolve 'IC-Light/iclight_sd15_fc.safetensors' → models/IC-Light/.
    new_lines += [
        '',
        '# Removation IC-Light fix — appended by notebook setup',
        'try:',
        "    add_model_folder_path('unet', '/content/ComfyUI/models')",
        'except Exception:',
        '    pass',
    ]
    with open(fp_file, 'w') as f:
        f.write('\n'.join(new_lines))
    print(f'✅ folder_paths.py patched: {patched}')

# 4g. INSPYRENET background removal model (used by RMBG node in furniture remover)
# Models/RMBG is now Drive-backed via MODEL_SUBDIRS, but pre-populating avoids
# a ~350 MB HuggingFace auto-download on the very first inference run.
inspyrenet_drive_dir = f'{DRIVE_MODELS}/RMBG/INSPYRENET'
inspyrenet_target    = f'{COMFYUI_PATH}/models/RMBG/INSPYRENET/inspyrenet.safetensors'
if os.path.lexists(inspyrenet_drive_dir) and not os.path.isdir(inspyrenet_drive_dir):
    os.remove(inspyrenet_drive_dir)
os.makedirs(inspyrenet_drive_dir, exist_ok=True)
inspyrenet_drive     = f'{inspyrenet_drive_dir}/inspyrenet.safetensors'
if os.path.exists(inspyrenet_target) and os.path.getsize(inspyrenet_target) > 100_000:
    print('✅ INSPYRENET ready')
elif os.path.exists(inspyrenet_drive) and os.path.getsize(inspyrenet_drive) > 100_000:
    print('✅ INSPYRENET (Drive cache already populated — symlink active)')
else:
    print('⏬ Downloading INSPYRENET...', end=' ', flush=True)
    os.system(f'wget -q -c -O "{inspyrenet_drive}" "https://huggingface.co/1038lab/inspyrenet/resolve/main/inspyrenet.safetensors"')
    if os.path.exists(inspyrenet_drive) and os.path.getsize(inspyrenet_drive) > 100_000:
        print('✅')
    else:
        print('❌ (will auto-download on first use instead)')

print('✅ All pre-start fixes applied')

# ── STEP 5: Custom nodes ───────────────────────────────────────
print()
print('━'*55)
print('STEP 5/9  Install custom nodes')
print('━'*55)
os.makedirs(NODES_PATH, exist_ok=True)

REQUIRED_NODES = [
    ('ComfyUI-Manager',             'https://github.com/ltdrdata/ComfyUI-Manager.git'),
    ('comfyui_controlnet_aux',      'https://github.com/Fannovel16/comfyui_controlnet_aux.git'),
    ('ComfyUI-Advanced-ControlNet', 'https://github.com/Kosinkadink/ComfyUI-Advanced-ControlNet.git'),
    ('comfyui-inpaint-nodes',       'https://github.com/Acly/comfyui-inpaint-nodes.git'),
    ('comfyui-lama-remover',        'https://github.com/Layer-norm/comfyui-lama-remover.git'),
    ('ComfyUI-IC-Light',            'https://github.com/huchenlei/ComfyUI-IC-Light.git'),
    ('comfyui-impact-pack',         'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git'),
    ('rgthree-comfy',               'https://github.com/rgthree/rgthree-comfy.git'),
    ('comfyui-kjnodes',             'https://github.com/kijai/ComfyUI-KJNodes.git'),
    ('ComfyUI_IPAdapter_plus',      'https://github.com/cubiq/ComfyUI_IPAdapter_plus.git'),
    ('was-node-suite-comfyui',      'https://github.com/WASasquatch/was-node-suite-comfyui.git'),
    ('ComfyUI-GGUF',                'https://github.com/city96/ComfyUI-GGUF.git'),
    ('ComfyUI-RMBG',                'https://github.com/1038lab/ComfyUI-RMBG.git'),
    ('comfyui-segment-anything',    'https://github.com/storyicon/comfyui_segment_anything.git'),
    ('ComfyUI-FluxKontext',         'https://github.com/lum3on/ComfyUI-FluxKontext.git'),
    ('comfyui-ollama',              'https://github.com/stavsap/comfyui-ollama.git'),
    ('ComfyUI-VideoHelperSuite',    'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git'),
    ('comfyui-segment-anything2',   'https://github.com/kijai/ComfyUI-segment-anything2.git'),
    ('comfyui-essentials',          'https://github.com/cubiq/ComfyUI_essentials.git'),
    ('ComfyUI-AnimateDiff-Evolved', 'https://github.com/Kosinkadink/ComfyUI-AnimateDiff-Evolved.git'),
    ('ComfyUI-MiniMax-Remover',     'https://github.com/MinusZoneAI/ComfyUI-MiniMax-Image-Remover.git'),
    ('ComfyUI-AILab',               'https://github.com/1038lab/ComfyUI-AILab-Nodes.git'),
]

# Force-clone: these nodes had failed Drive clones (empty/missing __init__.py).
# Code is always cloned fresh; models stay Drive-backed separately.
NODES_FORCE_CLONE = {'comfyui_controlnet_aux', 'ComfyUI-MiniMax-Remover'}

def _git_clone_with_retry(url, dest, retries=3):
    for attempt in range(retries):
        ret = os.system(f'git clone {url} {dest} -q 2>/dev/null')
        if ret == 0 and os.path.exists(dest):
            return True
        time.sleep(5)
    return False

for name, url in REQUIRED_NODES:
    drive_node = f'{DRIVE_NODES}/{name}'
    comfy_node = f'{NODES_PATH}/{name}'

    # Detect dummy directory (models but no Python code)
    models_backup = None
    if os.path.exists(comfy_node) and not os.path.islink(comfy_node) and os.path.isdir(comfy_node):
        has_code = any(fn.endswith('.py') for _, _, files in os.walk(comfy_node) for fn in files)
        if not has_code:
            print(f'  ⚠️ Dummy dir (models only) for {name} — backing up...')
            models_backup = f'/tmp/models_backup_{name}'
            if os.path.exists(models_backup):
                shutil.rmtree(models_backup)
            shutil.move(comfy_node, models_backup)

    if name in NODES_FORCE_CLONE:
        if os.path.islink(comfy_node):
            os.unlink(comfy_node)
        if os.path.exists(comfy_node) and not os.path.exists(f'{comfy_node}/.git'):
            shutil.rmtree(comfy_node)
        if not os.path.exists(comfy_node):
            print(f'  ⏬ {name} (git clone)...', end=' ', flush=True)
            ok = _git_clone_with_retry(url, comfy_node)
            print('✅' if ok else '❌')
        else:
            print(f'  ✅ {name} (clone exists)')

        # FIX #1 Part B: patch requirements.txt before pip runs it
        req_path = f'{comfy_node}/requirements.txt'
        if os.path.exists(req_path):
            with open(req_path) as f:
                req_text = f.read()
            changed = False
            if 'opencv-python\n' in req_text:
                req_text = req_text.replace('opencv-python\n', 'opencv-python-headless\n')
                changed = True
            if 'onnxruntime-gpu' in req_text:
                req_text = req_text.replace('onnxruntime-gpu', 'onnxruntime')
                changed = True
            if changed:
                with open(req_path, 'w') as f:
                    f.write(req_text)
                print(f'  🔧 {name}/requirements.txt patched (opencv-headless, onnxruntime)')
    else:
        def _has_init(p):
            return os.path.isfile(os.path.join(p, '__init__.py'))

        def _ensure_symlink(src_dir, dst_link):
            if os.path.islink(dst_link):
                os.unlink(dst_link)
            elif os.path.isdir(dst_link):
                shutil.rmtree(dst_link)
            elif os.path.lexists(dst_link):
                os.remove(dst_link)
            os.symlink(src_dir, dst_link)

        # If Drive node exists but has no __init__.py (failed clone), wipe and re-clone.
        if os.path.exists(drive_node) and not _has_init(drive_node):
            print(f'  ⚠️  {name} Drive clone is empty/broken — re-cloning to Drive...')
            shutil.rmtree(drive_node)
            if os.path.islink(comfy_node): os.unlink(comfy_node)

        if os.path.exists(comfy_node) and _has_init(comfy_node):
            print(f'  ✅ {name}')
        elif os.path.islink(comfy_node):
            # Dangling or broken symlink — remove it
            os.unlink(comfy_node)

        if not os.path.exists(comfy_node):
            if os.path.exists(drive_node):
                _ensure_symlink(drive_node, comfy_node)
                print(f'  🔗 {name} (Drive)')
            else:
                print(f'  ⏬ {name}...', end=' ', flush=True)
                ok = _git_clone_with_retry(url, drive_node)
                if ok:
                    _ensure_symlink(drive_node, comfy_node)
                    print('✅')
                else:
                    print('❌ (clone failed — will skip, non-critical)')

    if models_backup and os.path.exists(models_backup):
        print(f'  📦 Restoring models for {name}...')
        os.makedirs(comfy_node, exist_ok=True)
        for item in os.listdir(models_backup):
            src_i = os.path.join(models_backup, item)
            dst_i = os.path.join(comfy_node, item)
            if not os.path.exists(dst_i):
                shutil.move(src_i, dst_i)
        shutil.rmtree(models_backup)

    req = f'{comfy_node}/requirements.txt'
    if os.path.exists(req):
        os.system(f'pip install -q -r {req} 2>/dev/null || true')

# FIX #3: Append M_LSD_Lines — format-proof, works regardless of dict style
mlsd_path = f'{NODES_PATH}/comfyui_controlnet_aux/node_wrappers/mlsd.py'
if os.path.exists(mlsd_path):
    with open(mlsd_path, 'r') as f:
        mlsd_code = f.read()
    if 'M_LSD_Lines' not in mlsd_code:
        mlsd_code += (
            '\n# Legacy alias — Removation workflows use M_LSD_Lines node type\n'
            'NODE_CLASS_MAPPINGS["M_LSD_Lines"] = MLSD_Preprocessor\n'
            'NODE_DISPLAY_NAME_MAPPINGS["M_LSD_Lines"] = "M-LSD Lines"\n'
        )
        with open(mlsd_path, 'w') as f:
            f.write(mlsd_code)
        pyc_dir = os.path.join(os.path.dirname(mlsd_path), '__pycache__')
        if os.path.exists(pyc_dir):
            for pyc in os.listdir(pyc_dir):
                if 'mlsd' in pyc:
                    os.remove(os.path.join(pyc_dir, pyc))
        print('✅ M_LSD_Lines alias appended to mlsd.py (stale .pyc cleared)')
    else:
        print('✅ M_LSD_Lines already present in mlsd.py')
else:
    print('❌ mlsd.py not found — comfyui_controlnet_aux clone failed')

os.system(f'cd {NODES_PATH}/comfyui-impact-pack && python install.py 2>/dev/null || true')

# MiniMax: node was force-cloned fresh (code only). Symlink Drive models in.
# Drive stores models at: DRIVE_NODES/ComfyUI-MiniMax-Remover/models/
# Node loader expects them at: custom_nodes/ComfyUI-MiniMax-Remover/models/
_mx_node_dir  = f'{NODES_PATH}/ComfyUI-MiniMax-Remover'
_mx_drive_mdl = f'{DRIVE_NODES}/ComfyUI-MiniMax-Remover/models'
if os.path.isdir(_mx_node_dir) and not os.path.islink(_mx_node_dir):
    _mx_local_mdl = f'{_mx_node_dir}/models'
    if os.path.islink(_mx_local_mdl):
        os.unlink(_mx_local_mdl)
    elif os.path.isdir(_mx_local_mdl):
        # Merge any locally-written models to Drive, then replace with symlink
        for _mi in os.listdir(_mx_local_mdl):
            _ms = os.path.join(_mx_local_mdl, _mi)
            _md = os.path.join(_mx_drive_mdl, _mi)
            if not os.path.exists(_md):
                os.makedirs(os.path.dirname(_md), exist_ok=True)
                shutil.move(_ms, _md)
        shutil.rmtree(_mx_local_mdl)
    # Always ensure Drive dir exists before symlinking (Step 5b may not have run yet
    # on first session; makedirs is safe even if dir was created by Step 5b already).
    os.makedirs(_mx_drive_mdl, exist_ok=True)
    os.symlink(_mx_drive_mdl, _mx_local_mdl)
    print('  ✅ MiniMax: models/ → Drive symlink wired')
print('✅ All custom nodes ready')

# ── STEP 5b: MiniMax Remover models ───────────────────────────────────────
# MinimaxModelLoader hardcodes local_files_only=True and looks for models at
#   {custom_node_dir}/models/MiniMax-Remover/{vae,transformer,scheduler}/
# Because the node is symlinked to Drive, writing to DRIVE_NODES/.../models/
# makes the files immediately visible to the node at its internal path.
print()
print('━'*55)
print('STEP 5b  MiniMax Remover models (~2.6 GB, cached to Drive)')
print('━'*55)

MINIMAX_DRIVE = f'{DRIVE_NODES}/ComfyUI-MiniMax-Remover/models/MiniMax-Remover'
MINIMAX_HF    = 'zibojia/minimax-remover'
HF_BASE       = f'https://huggingface.co/{MINIMAX_HF}/resolve/main'

# (sub-dirs, minimum_valid_size_bytes)
MINIMAX_FILES = [
    ('scheduler/scheduler_config.json',                      50),
    ('vae/config.json',                                      50),
    ('vae/diffusion_pytorch_model.safetensors',       400_000_000),
    ('transformer/config.json',                              50),
    ('transformer/diffusion_pytorch_model.safetensors', 1_000_000_000),
]

minimax_all_ok = True
for rel_path, size_floor in MINIMAX_FILES:
    drive_p = f'{MINIMAX_DRIVE}/{rel_path}'
    os.makedirs(os.path.dirname(drive_p), exist_ok=True)
    if os.path.exists(drive_p) and os.path.getsize(drive_p) > size_floor:
        print(f'  ✅ {rel_path}')
    else:
        print(f'  ⏬ {rel_path}...', end=' ', flush=True)
        os.system(f'wget -q -c -O "{drive_p}" "{HF_BASE}/{rel_path}"')
        if os.path.exists(drive_p) and os.path.getsize(drive_p) > size_floor:
            print('✅')
        else:
            print('❌ (download failed — furniture remover auto-mode will not work)')
            minimax_all_ok = False

if minimax_all_ok:
    print('✅ MiniMax models ready (Drive-backed, symlinked into node)')
else:
    print('⚠️  MiniMax models incomplete — run again or upload manually to Drive')

# ── STEP 6: Models ───────────────────────────────────────────
print()
print('━'*55)
print('STEP 6/9  Download models (once to Drive)')
print('━'*55)

def get_model(subdir, filename, url, label, size_floor=100_000):
    drive_path = f'{DRIVE_MODELS}/{subdir}/{filename}'
    comfy_path = f'{COMFYUI_PATH}/models/{subdir}/{filename}'

    def _copy_to_comfy():
        if os.path.exists(drive_path) and not os.path.exists(comfy_path):
            os.makedirs(os.path.dirname(comfy_path), exist_ok=True)
            shutil.copy2(drive_path, comfy_path)

    if os.path.exists(comfy_path) and os.path.getsize(comfy_path) > size_floor:
        print(f'  ✅ {label}')
        return True
    if os.path.exists(drive_path) and os.path.getsize(drive_path) > size_floor:
        _copy_to_comfy()
        print(f'  ✅ {label}')
        return True
    print(f'  ⏬ {label}...', end=' ', flush=True)
    os.makedirs(os.path.dirname(drive_path), exist_ok=True)
    os.system(f'wget -q -c -O "{drive_path}" "{url}"')
    if os.path.exists(drive_path) and os.path.getsize(drive_path) > size_floor:
        _copy_to_comfy()
        print('✅')
        return True
    print('❌')
    return False

# ── Corrupt model validation ──────────────────────────────────────────────
# PyTorch .pth models (new format) are ZIP files internally.
# "PytorchStreamReader failed locating file data/0" = corrupt/partial download.
# Detect corruption via zip structure check; delete so get_model re-downloads.
import zipfile as _zf

def _pth_valid(path, min_bytes=1_000_000):
    """Return True if path is a valid new-format PyTorch ZIP .pth file."""
    if not (os.path.isfile(path) and os.path.getsize(path) >= min_bytes):
        return False
    try:
        with _zf.ZipFile(path, 'r') as _z:
            _names = _z.namelist()
            # New-format .pth always has 'archive/data.pkl' or 'data/0'
            return any(n.startswith('archive/') or n.startswith('data/') for n in _names)
    except (_zf.BadZipFile, Exception):
        return False

# Validate upscale models only (small enough to check quickly; safetensors skip).
# Only scan Drive path — ComfyUI/models/upscale_models is a symlink to the same location;
# iterating both would produce duplicate output and a second failed os.remove().
for _udir in [f'{DRIVE_MODELS}/upscale_models']:
    if not os.path.isdir(_udir):
        continue
    for _uf in sorted(os.listdir(_udir)):
        if not _uf.endswith('.pth'):
            continue
        _up = os.path.join(_udir, _uf)
        if not _pth_valid(_up, min_bytes=1_000_000):
            _kb = os.path.getsize(_up)//1024 if os.path.isfile(_up) else 0
            print(f'  🗑️  {_uf} invalid/corrupt ({_kb}KB, bad zip) — deleting to force re-download')
            if os.path.isfile(_up):
                os.remove(_up)

get_model('checkpoints', 'sd_xl_base_1.0.safetensors',
    'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors',
    'SDXL Base 1.0')
get_model('checkpoints', 'photon_v1.safetensors',
    'https://huggingface.co/digiplay/Photon_v1/resolve/main/photon_v1.safetensors',
    'photon_v1 [Style Transfer]')
get_model('checkpoints', '1_5photon_v1.safetensors',
    'https://huggingface.co/digiplay/Photon_v1/resolve/main/photon_v1.safetensors',
    '1_5photon_v1 [IC-Light]')
get_model('checkpoints', 'juggernautXL_version6Rundiffusion.safetensors',
    'https://huggingface.co/RunDiffusion/Juggernaut-XL-v6/resolve/main/juggernautXL_version6Rundiffusion.safetensors',
    'JuggernautXL v6 [Outpaint]')
get_model('checkpoints', 'realvisxlV40.safetensors',
    'https://huggingface.co/SG161222/RealVisXL_V4.0/resolve/main/RealVisXL_V4.0.safetensors',
    'RealVisXL V4.0')
get_model('controlnet', 'control_v11p_sd15_canny.pth',
    'https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_canny.pth',
    'ControlNet Canny SD1.5')

# FIX #5: MLSD in controlnet/1.5/ + auto-migrate old copy
_mlsd_target = f'{DRIVE_MODELS}/controlnet/1.5/control_v11p_sd15_mlsd_fp16.safetensors'
_mlsd_old    = f'{DRIVE_MODELS}/controlnet/control_v11p_sd15_mlsd_fp16.safetensors'
os.makedirs(f'{DRIVE_MODELS}/controlnet/1.5', exist_ok=True)
if os.path.exists(_mlsd_target) and os.path.getsize(_mlsd_target) > 100_000:
    print('  ✅ MLSD ControlNet SD1.5 [Style Transfer]')
elif os.path.exists(_mlsd_old) and os.path.getsize(_mlsd_old) > 100_000:
    shutil.copy2(_mlsd_old, _mlsd_target)
    print('  ✅ MLSD migrated → controlnet/1.5/')
else:
    print('  ⏬ MLSD ControlNet SD1.5...', end=' ', flush=True)
    os.system(f'wget -q -c -O "{_mlsd_target}" "https://huggingface.co/lllyasviel/ControlNet-v1-1/resolve/main/control_v11p_sd15_mlsd_fp16.safetensors"')
    if os.path.exists(_mlsd_target) and os.path.getsize(_mlsd_target) > 100_000:
        print('✅')
    else:
        print('❌  — upload control_v11p_sd15_mlsd_fp16.safetensors to Drive/RemovationAI/models/controlnet/1.5/')

get_model('controlnet', 'controlnet-canny-sdxl.safetensors',
    'https://huggingface.co/xinsir/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors',
    'ControlNet Canny SDXL')
get_model('controlnet', 'control-lora-canny-rank256.safetensors',
    'https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-canny-rank256.safetensors',
    'Control-LoRA Canny rank256')
get_model('controlnet', 'control-lora-depth-rank256.safetensors',
    'https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-depth-rank256.safetensors',
    'Control-LoRA Depth rank256')
get_model('controlnet', 'control-lora-sketch-rank256.safetensors',
    'https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-sketch-rank256.safetensors',
    'Control-LoRA Sketch rank256')
get_model('controlnet', 'control-lora-recolor-rank256.safetensors',
    'https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank256/control-lora-recolor-rank256.safetensors',
    'Control-LoRA Recolor rank256')
get_model('upscale_models', 'RealESRGAN_x4plus.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    'RealESRGAN x4plus')
get_model('upscale_models', 'RealESRGAN_x4.pth',
    'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth',
    'RealESRGAN x4 general')
get_model('vae', 'sdxl_vae.safetensors',
    'https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors',
    'SDXL VAE fp16-fix')
get_model('ipadapter', 'ip-adapter-plus_sdxl_vit-h.safetensors',
    'https://huggingface.co/h94/IP-Adapter/resolve/main/sdxl_models/ip-adapter-plus_sdxl_vit-h.safetensors',
    'IP-Adapter Plus SDXL')
get_model('clip_vision', 'CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors',
    'https://huggingface.co/h94/IP-Adapter/resolve/main/models/image_encoder/model.safetensors',
    'CLIP-ViT-H-14')
get_model('IC-Light', 'iclight_sd15_fc.safetensors',
    'https://huggingface.co/huchenlei/IC-Light/resolve/main/iclight_sd15_fc.safetensors',
    'IC-Light fc')
get_model('IC-Light', 'iclight_sd15_fbc.safetensors',
    'https://huggingface.co/huchenlei/IC-Light/resolve/main/iclight_sd15_fbc.safetensors',
    'IC-Light fbc (background)')

# IC-Light path fix: Google Drive FUSE does NOT support os.symlink() inside it
# (raises OSError: Operation not supported). Instead, folder_paths.py is patched
# in Step 4f to register models/ as an extra unet search dir, so ComfyUI resolves
# 'IC-Light/iclight_sd15_fc.safetensors' to models/IC-Light/ — no symlinks needed.
# Also guard with size check: failed wget leaves a 0-byte file that fools os.path.exists.
_ic_src_dir = f'{COMFYUI_PATH}/models/IC-Light'
for _ic_fname in ['iclight_sd15_fc.safetensors', 'iclight_sd15_fbc.safetensors']:
    _ic_path = f'{_ic_src_dir}/{_ic_fname}'
    if os.path.exists(_ic_path) and os.path.getsize(_ic_path) > 100_000:
        print(f'  ✅ IC-Light/{_ic_fname} (Drive-backed, accessible via folder_paths)')
    else:
        print(f'  ⚠️  IC-Light/{_ic_fname} missing or download failed — delete partial file and re-run')
get_model('sam2', 'sam2_hiera_small.pt',
    'https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt',
    'SAM2 Hiera Small')
get_model('depth_anything_v2', 'depth_anything_v2_vits.pth',
    'https://huggingface.co/depth-anything/Depth-Anything-V2-Small/resolve/main/depth_anything_v2_vits.pth',
    'Depth Anything V2')
get_model('grounding-dino', 'groundingdino_swint_ogc.pth',
    'https://huggingface.co/ShilongLiu/GroundingDINO/resolve/main/groundingdino_swint_ogc.pth',
    'GroundingDINO')
get_model('inpaint', 'fooocus_inpaint_head.pth',
    'https://huggingface.co/lllyasviel/fooocus_inpaint/resolve/main/fooocus_inpaint_head.pth',
    'Fooocus Inpaint Head')
get_model('clip', 'clip_l.safetensors',
    'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors',
    'CLIP-L')
get_model('clip', 't5xxl_fp8_e4m3fn.safetensors',
    'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors',
    'T5-XXL fp8')
get_model('clip_vision', 'sigclip_vision_patch14_384.safetensors',
    'https://huggingface.co/Comfy-Org/sigclip_vision_384/resolve/main/sigclip_vision_patch14_384.safetensors',
    'SigCLIP Vision')

print()
print('  Gated models (upload to Drive once to unlock features):')
GATED = [
    ('diffusion_models', 'flux1-dev-fp8.safetensors',                  'Furniture Staging'),
    ('vae',              'ae.safetensors',                             'Furniture Staging'),
    ('style_models',     'flux1-redux-dev.safetensors',                'Furniture Staging'),
    ('loras',            'flux_realism_lora.safetensors',              'Furniture Staging'),
    ('diffusion_models', 'qwen_image_edit_2509_fp8_e4m3fn.safetensors','Material Changer'),
    ('clip',             'qwen_2.5_vl_7b_fp8_scaled.safetensors',     'Material Changer'),
    ('vae',              'qwen_image_vae.safetensors',                 'Material Changer'),
    ('loras',            'Qwen-Edit-2509-Multiple-angles.safetensors', 'Material Changer'),
    ('unet',             'qwen-image-edit-2509.Q4_K_M.gguf',          'Material Changer'),
]
for subdir, fname, feature in GATED:
    p = f'{DRIVE_MODELS}/{subdir}/{fname}'
    icon = '✅' if (os.path.exists(p) and os.path.getsize(p) > 100_000) else '❌'
    print(f'  {icon} {fname}  [{feature}]')

# ── STEP 7: Start ComfyUI ──────────────────────────────────────
print()
print('━'*55)
print('STEP 7/9  Start ComfyUI')
print('━'*55)

# FIX #1 Part C: reinstall deps every session (wiped on restart)
# Show actual errors — not silenced
print('📦 Reinstalling comfyui_controlnet_aux deps...')
aux_req = f'{NODES_PATH}/comfyui_controlnet_aux/requirements.txt'
if os.path.exists(aux_req):
    result = subprocess.run(['pip', 'install', '-r', aux_req], capture_output=True, text=True)
    if result.returncode != 0:
        print('⚠️  controlnet_aux pip errors (last 10 lines):')
        for ln in result.stderr.strip().splitlines()[-10:]:
            print('   ', ln)
    else:
        print('✅ controlnet_aux requirements installed')

# Re-enforce headless after requirements (requirements re-installs full opencv)
os.system('pip uninstall -q -y opencv-python 2>/dev/null || true')
os.system('pip install -q opencv-python-headless 2>/dev/null || true')
os.system('pip install -q controlnet-aux timm einops insightface 2>/dev/null || true')
print('✅ opencv-headless enforced')

# Retry depth model copy (in case Step 4 ran before node was cloned)
# Retry flat path (Step 4c fallback — node was not yet cloned then)
depth_ckpts_base_r = f'{NODES_PATH}/comfyui_controlnet_aux/ckpts'
depth_nested_dir_r = f'{depth_ckpts_base_r}/depth-anything/Depth-Anything-V2-Small'
for _d_target in [f'{depth_ckpts_base_r}/depth_anything_v2_vits.pth',
                  f'{depth_nested_dir_r}/depth_anything_v2_vits.pth']:
    os.makedirs(os.path.dirname(_d_target), exist_ok=True)
    if not os.path.exists(_d_target) and os.path.exists(depth_src):
        shutil.copy2(depth_src, _d_target)
depth_target = f'{depth_ckpts_base_r}/depth_anything_v2_vits.pth'

os.system('pkill -f "python main.py" 2>/dev/null')
time.sleep(4)

# FIX (v2.1): Use python -u to disable output buffering.
# Without -u, Python buffers stdout when writing to a file. If ComfyUI crashes
# early, the buffer is never flushed, the log stays empty, and we see no error.
comfy_log = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(
    ['python', '-u', 'main.py',
     '--listen', '0.0.0.0', '--port', '8188',
     '--preview-method', 'none', '--lowvram'],
    cwd=COMFYUI_PATH, stdout=comfy_log, stderr=comfy_log
)

# FIX (v2.1): Timeout 600s (10 min) — fresh node clones may download models on first load
print('Loading all custom nodes (up to 10 min on first run)', end='', flush=True)
comfy_ready = False
for i in range(200):
    time.sleep(3)
    if i % 20 == 19:
        print(f' [{(i+1)*3}s]', end='', flush=True)
    else:
        print('.', end='', flush=True)

    # Check if process died
    if proc.poll() is not None:
        print(f'\n❌ ComfyUI process died (exit code {proc.poll()})')
        break

    try:
        if requests.get('http://localhost:8188/system_stats', timeout=2).status_code == 200:
            comfy_ready = True
            print(' ✅')
            break
    except:
        pass

comfy_log.flush()

if not comfy_ready:
    if proc.poll() is None:
        print('\n❌ ComfyUI timed out (still running but not responding)')
    print('\n=== Last 80 lines of /content/comfyui.log ===')
    result = subprocess.run(['tail', '-80', '/content/comfyui.log'], capture_output=True, text=True)
    print(result.stdout or '(log is empty — process may have crashed before writing anything)')
    print('\n=== controlnet_aux / ImportError lines ===')
    result2 = subprocess.run(['grep', '-iE', 'controlnet_aux|M_LSD|hint_image|cv2|ImportError|libGL|Failed to import|Traceback|Error', '/content/comfyui.log'],
                             capture_output=True, text=True)
    print(result2.stdout or '(no matching lines)')
    print('\nTo investigate: add a new cell and run:')
    print('  !ps aux | grep main.py')
    print('  !cat /content/comfyui.log')
    raise RuntimeError('ComfyUI failed to start — see log above')

# ── STEP 8: Cloudflare tunnel ────────────────────────────────
print()
print('━'*55)
print('STEP 8/9  Cloudflare tunnel')
print('━'*55)
if not os.path.exists('/content/cloudflared'):
    os.system('wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared')

os.system('pkill -f cloudflared 2>/dev/null')
time.sleep(2)
tunnel_log = open('/content/tunnel.log', 'w')
subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=tunnel_log, stderr=tunnel_log
)

print('Connecting', end='', flush=True)
tunnel_url = None
for _ in range(30):
    time.sleep(2)
    print('.', end='', flush=True)
    try:
        with open('/content/tunnel.log') as f:
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', f.read())
        if m:
            tunnel_url = m.group(0)
            print(' ✅')
            break
    except:
        pass

def _ping():
    while True:
        try: requests.get('http://localhost:8188/system_stats', timeout=5)
        except: pass
        time.sleep(30)
threading.Thread(target=_ping, daemon=True).start()

# ── STEP 9: Final verification ───────────────────────────────
print()
print('━'*55)
print('STEP 9/9  Final verification')
print('━'*55)

critical_nodes = {
    'Room Changer / Ceiling':  ['SAMLoader', 'CLIPSegDetectorProvider', 'CLIPSeg', 'ImpactSimpleDetectorSEGS', 'SegsToCombinedMask'],
    'Furniture Removal':       ['LamaRemoverIMG'],
    'IC-Light Relighting':     ['LoadAndApplyICLight', 'ImageResizeKJ'],
    'Outpaint':                ['INPAINT_LoadFooocusInpaint', 'INPAINT_ApplyFooocusInpaint'],
    'Style Transfer':          ['IPAdapterUnifiedLoader', 'M_LSD_Lines'],
    'Scene Understanding':     ['DepthAnythingV2Preprocessor', 'GroundingDinoModelLoader'],
    'Material Changer':        ['UnetLoaderGGUF'],
    'Background Removal':      ['RMBG'],
    'Scene Analysis (SAM2)':   ['SAM2ModelLoader (segment anything2)', 'GroundingDinoSAM2Segment (segment anything2)'],
    'Animated Lighting':       ['ADE_AnimateDiffLoaderGen1', 'AlignYourStepsScheduler'],
    'Essentials Utility':      ['ImageResize+', 'GrowMaskWithBlur'],
    'MiniMax Erase':           ['MinimaxImageRemover'],
    'AILab Placement':         ['AILab_Preview'],
}

try:
    node_data = requests.get('http://localhost:8188/object_info', timeout=10).json()
    print('Node status:')
    all_ok = True
    for feature, nodes in critical_nodes.items():
        missing = [n for n in nodes if n not in node_data]
        if missing:
            print(f'  ❌ {feature}: missing {missing}')
            all_ok = False
        else:
            print(f'  ✅ {feature}')

    if 'M_LSD_Lines' not in node_data:
        print()
        print('  ❌ M_LSD_Lines still missing — controlnet_aux errors from log:')
        r = subprocess.run(['grep', '-iE', 'Failed to import|controlnet_aux|hint_image|cv2|ImportError|libGL',
                           '/content/comfyui.log'], capture_output=True, text=True)
        print(r.stdout or '  (no matching lines in log)')

    try:
        sam_data = requests.get('http://localhost:8188/object_info/SAMLoader', timeout=5).json()
        sam_models = sam_data.get('SAMLoader', {}).get('input', {}).get('required', {}).get('model_name', [[]])
        if sam_models and len(sam_models[0]) > 0:
            print(f'  ✅ SAMLoader sees models: {sam_models[0]}')
        else:
            print('  ❌ SAMLoader model list empty — Room Changer will fail')
            all_ok = False
    except:
        pass

    print()
    print('✅ ALL NODES READY' if all_ok else '⚠️  Some nodes missing — see above')
except Exception as e:
    print(f'Verification failed: {e}')

if tunnel_url:
    try:
        resp = requests.get(f'{tunnel_url}/system_stats', timeout=20,
                            headers={'bypass-tunnel-reminder': 'true'})
        d = resp.json()
        dev = d.get('devices', [{}])[0]
        vfree  = dev.get('vram_free',  0) / 1024**3
        vtotal = dev.get('vram_total', 0) / 1024**3
        verified = f'Verified: {vfree:.1f} GB free / {vtotal:.1f} GB total'
    except:
        verified = 'Not yet reachable externally — paste URL anyway'

    print()
    print('╔' + '═'*54 + '╗')
    print('║           REMOVATION.AI COLAB READY                    ║')
    print('╠' + '═'*54 + '╣')
    print('║  1. Copy this URL:                                      ║')
    print(f'║     {tunnel_url:<52} ║')
    print('║                                                         ║')
    print('║  2. Admin panel -> Settings -> GPU -> Cloud URL         ║')
    print('║     Paste URL -> Mode: cloud -> Save                    ║')
    print('║                                                         ║')
    print(f'║     {verified:<52} ║')
    print('╚' + '═'*54 + '╝')
else:
    print('❌ No tunnel URL — run: !cat /content/tunnel.log')

## 🔍 Debug Cell — run this if ComfyUI failed to start

In [ ]:
# Run this cell separately if the main cell failed.
import subprocess, os, requests

print('=== Process check ===')
r = subprocess.run(['ps', 'aux'], capture_output=True, text=True)
procs = [l for l in r.stdout.split('\n') if 'main.py' in l]
print('ComfyUI running:', procs or 'NO — process is dead')

print('\n=== ComfyUI reachable? ===')
try:
    print(requests.get('http://localhost:8188/system_stats', timeout=3).json())
except Exception as e:
    print('Not reachable:', e)

print('\n=== Full /content/comfyui.log ===')
if os.path.exists('/content/comfyui.log'):
    content = open('/content/comfyui.log').read()
    print(f'Size: {len(content)} bytes')
    print(content[-8000:] if len(content) > 8000 else content or '(empty)')
else:
    print('Log file does not exist')

print('\n=== Errors only ===')
r2 = subprocess.run(
    ['grep', '-niE', 'Error|Exception|Traceback|Failed|killed|killed|Killed|cannot|ImportError|SyntaxError',
     '/content/comfyui.log'],
    capture_output=True, text=True
)
print(r2.stdout[-4000:] or '(no error lines)')


## ♻️ Restart Cell — run after making changes

Run **Cell 1 first** (once per Colab session) to install everything.  
Whenever you make a small change and need a **new tunnel URL**, just run this cell — it restarts ComfyUI and the tunnel without re-downloading anything.

In [ ]:
# ╭──────────────────────────────────────────────────────────────╮
# │  RESTART — kills old ComfyUI + tunnel, starts fresh, new URL │
# │  Cell 1 must have been run this session before using this.   │
# ╰──────────────────────────────────────────────────────────────╯
import os, re, time, subprocess, threading, requests

COMFYUI_PATH = '/content/ComfyUI'

# ── 1. Stop old processes ─────────────────────────────────────────────────────
print('Stopping old processes...')
os.system('pkill -f "python main.py" 2>/dev/null')
os.system('pkill -f cloudflared 2>/dev/null')
time.sleep(3)

# ── 2. Start ComfyUI ──────────────────────────────────────────────────────────
comfy_log = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(
    ['python', '-u', 'main.py',
     '--listen', '0.0.0.0', '--port', '8188',
     '--preview-method', 'none', '--lowvram'],
    cwd=COMFYUI_PATH, stdout=comfy_log, stderr=comfy_log
)

print('Starting ComfyUI', end='', flush=True)
comfy_ready = False
for _i in range(200):
    time.sleep(3)
    print('.', end='', flush=True)
    if proc.poll() is not None:
        comfy_log.flush()
        print(f'\n❌ ComfyUI died (exit {proc.poll()})')
        print(open('/content/comfyui.log').read()[-3000:])
        raise RuntimeError('ComfyUI failed to start — see log above')
    try:
        if requests.get('http://localhost:8188/system_stats', timeout=2).status_code == 200:
            comfy_ready = True
            print(' ✅')
            break
    except Exception:
        pass

comfy_log.flush()
if not comfy_ready:
    print('\n❌ Timed out waiting for ComfyUI')
    print(open('/content/comfyui.log').read()[-3000:])
    raise RuntimeError('ComfyUI did not respond — see log above')

# ── 3. Start Cloudflare tunnel ────────────────────────────────────────────────
if not os.path.exists('/content/cloudflared'):
    os.system(
        'wget -q https://github.com/cloudflare/cloudflared/releases/latest/'
        'download/cloudflared-linux-amd64 -O /content/cloudflared '
        '&& chmod +x /content/cloudflared'
    )

tunnel_log = open('/content/tunnel.log', 'w')
subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8188'],
    stdout=tunnel_log, stderr=tunnel_log
)

print('Getting tunnel URL', end='', flush=True)
tunnel_url = None
for _ in range(30):
    time.sleep(2)
    print('.', end='', flush=True)
    try:
        with open('/content/tunnel.log') as f:
            m = re.search(r'https://[\w-]+\.trycloudflare\.com', f.read())
        if m:
            tunnel_url = m.group(0)
            print(' ✅')
            break
    except Exception:
        pass

# ── 4. Keep-alive ping (prevents Colab idle disconnect) ──────────────────────
def _ping():
    while True:
        try:
            requests.get('http://localhost:8188/system_stats', timeout=5)
        except Exception:
            pass
        time.sleep(30)
threading.Thread(target=_ping, daemon=True).start()

# ── 5. Print URL ──────────────────────────────────────────────────────────────
if tunnel_url:
    try:
        resp = requests.get(
            f'{tunnel_url}/system_stats', timeout=20,
            headers={'bypass-tunnel-reminder': 'true'}
        )
        dev     = resp.json().get('devices', [{}])[0]
        vfree   = dev.get('vram_free',  0) / 1024**3
        vtotal  = dev.get('vram_total', 0) / 1024**3
        verified = f'{vfree:.1f} GB free / {vtotal:.1f} GB total'
    except Exception:
        verified = 'tunnel reachable — paste URL to update dashboard'

    W = 58
    print()
    print('╔' + '═'*W + '╗')
    print('║' + '  REMOVATION.AI  —  NEW TUNNEL URL'.center(W) + '║')
    print('╠' + '═'*W + '╣')
    print('║' + f'  {tunnel_url}'.ljust(W) + '║')
    print('╠' + '═'*W + '╣')
    print('║' + '  Settings → GPU → Cloud URL → Paste → Save'.ljust(W) + '║')
    print('║' + f'  {verified}'.ljust(W) + '║')
    print('╚' + '═'*W + '╝')
else:
    print('❌ No tunnel URL found — run:  !cat /content/tunnel.log')
